# Competition-Conditioned Narrative Evolution in Online Social Networks
## ADA Course Project — Final Code Notebook

**Team:** The Outliers  
**Members:** Anuj Srijan (PES1UG24AM234), Mallikarjuna Rao R.V. (PES1UG24AM155), Deepesh M (PES1UG24AM150)

This notebook is the organised execution record for the primary analysis and the Dataset 3 reduced external evaluation. It separates the original analysis from external-replication outputs. The original project files and results should remain unchanged.

### Important execution notes
- Designed for **Google Colab + Google Drive**. Update `ROOT` if your project folder is stored elsewhere.
- The notebook calls project `.py` scripts stored in the project root. Keep those scripts, `config.py`, `common.py`, and the `data/`, `data_rep/`, `results/`, `results_rep/`, `figs/`, and `figs_rep/` folders alongside this notebook.
- Dataset 3 raw acquisition/extraction is large (about 44 GB extracted). Do not rerun acquisition unless the raw source files are available and a fresh reconstruction is required.
- Existing processed files and result CSVs are the reproducibility checkpoints. Heavy cells are labelled before execution.
- Preliminary Dataset 3 competition scripts (`25_...` and `26_...`) are retained in the project history but are **not used for final inference**. The corrected matched-refit test is Step 27.


## 1. Environment setup and project paths
Run this first in Google Colab. This cell mounts Drive and sets the working directory.

In [ ]:
   from google.colab import drive
   drive.mount('/content/drive')
   %cd "/content/drive/MyDrive/ADA Jackfruit"
   !ls
   !pip install -q sentence-transformers pyarrow
   !python patch_config.py

## 2. Primary study — original analysis pipeline

The following commands execute the original project pipeline using the scripts and data in the project root. They reproduce the primary study, not Dataset 3.

**Prerequisites:** `data/tweets.parquet`, `config.py`, `common.py`, and the original scripts (`02_embed_cluster.py` through `11_summary.py`) must be present.

**Note:** These commands can be computationally expensive. If the original results are already final, do not rerun them merely to prepare the submission notebook.


In [ ]:
   !python 02_embed_cluster.py --tweets data/tweets.parquet --backend st --k 4
   !python 03_label_agreement.py

In [ ]:
!python 04_targets.py
!python 06_pressure.py
!python 05_baseline.py --targets drift_db abs_sent_change abs_stance_change entity_turnover abs_frag_change abs_share_change abs_merge_change

In [ ]:
!python 07_main_test.py --target drift_db --n-placebo 500

In [ ]:
!for t in abs_sent_change abs_stance_change entity_turnover abs_frag_change abs_share_change abs_merge_change; do python 07_main_test.py --target $t --n-placebo 500; done
!python 07_main_test.py --target drift_db --n-placebo 500 --test-start 2022-11-01
!python 07_main_test.py --target drift_db --n-placebo 500 --test-start 2023-03-01
!python 07_main_test.py --target abs_stance_change --n-placebo 500 --test-start 2022-11-01
!python 08_classification.py --n-placebo 200
!python 11_summary.py

In [ ]:
!cp -r results results_primary_full
!cp -r figs figs_primary_full

### 2.1 Composition-control analysis

In [ ]:
!python 06b_composition_controls.py
!python 07_main_test.py --target drift_db --n-placebo 500 --extra-base shift_country,shift_channel --tag "competition+compctrl"
!python 07_main_test.py --target drift_db --n-placebo 500 --extra-base shift_country,shift_channel --tag "competition+compctrl" --test-start 2022-11-01

In [ ]:
!cp -r results results_compctrl
!cp -r figs figs_compctrl

### 2.2 Additional original-data checks

The original notebook also included a non-English extension and diffusion-proxy analyses. These are exploratory extensions and should be run only when the corresponding data and scripts are present.

In [ ]:
!wget -O /content/dataset_zenodo.jsonl "https://zenodo.org/records/8043459/files/dataset_zenodo.jsonl?download=1"
!NARR_RUN=_rep python 01_load_dataset.py --src /content/dataset_zenodo.jsonl --out data_rep/tweets.parquet --lang nonen

In [ ]:
!python -c "import pandas as pd; d=pd.read_parquet('data_rep/tweets.parquet'); print(len(d)); print(d.lang.value_counts().head()); print(d.text_en.sample(8, random_state=1).tolist())"

In [ ]:
!NARR_RUN=_rep python 02_embed_cluster.py --tweets data_rep/tweets.parquet --lang nonen --backend st --frac 0.5 --centroids data/narrative_centroids.npy
!NARR_RUN=_rep python 04_targets.py
!NARR_RUN=_rep python 06_pressure.py
!NARR_RUN=_rep python 07_main_test.py --target drift_db --n-placebo 500

In [ ]:
!cp -r results_rep results_rep_backup
!cp -r data_rep/*.parquet /content/ 2>/dev/null; ls results_rep

In [ ]:
!for t in d_volume d_breadth d_geo; do python 07_main_test.py --target $t --tag transformation->diffusion --n-placebo 500 --extra drift_db,abs_sent_change,abs_stance_change,entity_turnover,abs_frag_change,abs_merge_change --extra2 ""; done
!python 11_summary.py

## 3. Dataset 3 — reduced external evaluation

Dataset 3 uses tweet-level records from the BwandoWando Ukraine–Russian conflict Twitter dataset. The external pipeline is deliberately isolated under `data_rep/`, `results_rep/`, and `figs_rep/`.

**Important methodological boundary:** Dataset 3 does not provide all variables required by the original six-component competition score. The external analysis therefore uses a reduced three-component measure (semantic similarity, relative growth, and shared-user network proximity). Unavailable stance, sentiment, reach, and geographic variables are not fabricated.


### 3.1 Acquisition and sample preparation
The raw dataset is large. The following setup cell installs KaggleHub; the acquisition/adaptation scripts are external project files. Run only if reconstructing the sampled dataset from raw CSVs.

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "bwandowando/ukraine-russian-crisis-twitter-dataset-1-2-m-rows"
)

print("Path to dataset files:", path)

In [ ]:
import os

print("Dataset path:", path)

total_size = 0
file_count = 0

for root, dirs, files in os.walk(path):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_size += size
        file_count += 1

        print(f"{file} | {size / (1024**2):.2f} MB")

print("\nTotal files:", file_count)
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
import glob
import os

all_files = glob.glob(path + "/**/*", recursive=True)

data_files = [
    f for f in all_files
    if os.path.isfile(f)
    and f.lower().endswith((".csv", ".csv.gz", ".gzip", ".parquet"))
]

print("Data files found:", len(data_files))

for f in data_files[:30]:
    print(os.path.basename(f))

In [ ]:
import pandas as pd
import os

for f in data_files[:3]:
    print("\nFILE:", os.path.basename(f))

    try:
        sample = pd.read_csv(f, nrows=5, compression="infer")
        print("Columns:", sample.columns.tolist())
        display(sample.head(2))
    except Exception as e:
        print("Error:", e)

### 3.2 External preprocessing and embeddings

These commands rely on the project scripts `12_adapt_external_batch.py`, `13_preprocess_external.py`, and `14_embed_external.py`. The adapter creates the sampled external Parquet; preprocessing applies the project's text-cleaning and eligibility rules; embedding uses `sentence-transformers/all-MiniLM-L6-v2`.

**Resource note:** Embedding is GPU-preferred and produces approximately 1.1 GB of float32 vectors. The final processed sample and embeddings are already generated in the project checkpoint; rerun only if needed.


In [ ]:
!python "/content/drive/MyDrive/ADA Jackfruit/12_adapt_external_batch.py" \
 --manifest "/content/drive/MyDrive/ADA Jackfruit/data_rep/dataset3_csv_manifest.txt" \
 --original "/content/drive/MyDrive/ADA Jackfruit/data/tweets.parquet" \
 --out "/content/drive/MyDrive/ADA Jackfruit/data_rep/tweets_external_sample.parquet" \
 --fraction 0.03 --seed 42 --chunk-size 100000 --memory 4GB

In [ ]:
!python "/content/drive/MyDrive/ADA Jackfruit/13_preprocess_external.py" \
 --src "/content/drive/MyDrive/ADA Jackfruit/data_rep/tweets_external_sample.parquet" \
 --out-dir "/content/drive/MyDrive/ADA Jackfruit/data_rep"

In [ ]:
!python "/content/drive/MyDrive/ADA Jackfruit/14_embed_external.py" \
 --input "/content/drive/MyDrive/ADA Jackfruit/data_rep/external_unique_texts.parquet" \
 --out-dir "/content/drive/MyDrive/ADA Jackfruit/data_rep" \
 --chunk-size 25000 \
 --batch-size 256

### 3.3 Validate and assign frozen narratives

The original four narrative centroids are reused. K-means is not retrained on Dataset 3. The next cells validate the embedding and label-transfer artefacts created by the external scripts.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
CHUNK_DIR = ROOT / "data_rep" / "embedding_chunks"
TEXT_FILE = ROOT / "data_rep" / "external_unique_texts.parquet"

EXPECTED_TEXTS = 732_096
CHUNK_SIZE = 25_000
EXPECTED_DIM = 384

# Load unique-text count
texts = pd.read_parquet(TEXT_FILE)
assert len(texts) == EXPECTED_TEXTS, f"Unexpected text count: {len(texts)}"

# Validate all embedding chunks
files = sorted(CHUNK_DIR.glob("embedding_*.npy"))

assert len(files) == 30, f"Expected 30 chunks, found {len(files)}"

total_rows = 0
all_finite = True
norm_min = float("inf")
norm_max = 0.0

for i, file in enumerate(files):
    arr = np.load(file, mmap_mode="r")

    expected_rows = min(
        CHUNK_SIZE,
        EXPECTED_TEXTS - i * CHUNK_SIZE
    )

    assert arr.shape == (expected_rows, EXPECTED_DIM), (
        f"Incorrect shape in {file.name}: {arr.shape}"
    )

    assert arr.dtype == np.float32, (
        f"Incorrect dtype in {file.name}: {arr.dtype}"
    )

    finite = np.isfinite(arr).all()
    all_finite = all_finite and finite
    assert finite, f"Non-finite values in {file.name}"

    norms = np.linalg.norm(arr, axis=1)
    norm_min = min(norm_min, float(norms.min()))
    norm_max = max(norm_max, float(norms.max()))

    total_rows += arr.shape[0]

    print(
        f"{file.name}: {arr.shape}, "
        f"finite={finite}, "
        f"norm range=({norms.min():.5f}, {norms.max():.5f})"
    )

assert total_rows == EXPECTED_TEXTS
assert all_finite
assert abs(norm_min - 1.0) < 0.01
assert abs(norm_max - 1.0) < 0.01

print("\n=== VALIDATION SUCCESSFUL ===")
print("Total embeddings:", total_rows)
print("Dimensions:", EXPECTED_DIM)
print("Data type: float32")
print("All values finite:", all_finite)
print("Overall norm range:", norm_min, norm_max)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import os

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"
CHUNK_DIR = REP / "embedding_chunks"

TEXT_FILE = REP / "external_unique_texts.parquet"
OUT_FILE = REP / "emb_external.npy"
TEMP_FILE = REP / "emb_external.tmp.npy"

EXPECTED_TEXTS = 732_096
DIM = 384
CHUNK_SIZE = 25_000

# Confirm the source corpus order and size
texts = pd.read_parquet(TEXT_FILE)
assert len(texts) == EXPECTED_TEXTS

# Create a memory-mapped output matrix
emb = np.lib.format.open_memmap(
    TEMP_FILE,
    mode="w+",
    dtype=np.float32,
    shape=(EXPECTED_TEXTS, DIM)
)

# Copy chunks sequentially
position = 0

for i in range(30):
    chunk_file = CHUNK_DIR / f"embedding_{i:04d}.npy"
    assert chunk_file.exists(), f"Missing: {chunk_file}"

    chunk = np.load(chunk_file, mmap_mode="r")

    expected_rows = min(
        CHUNK_SIZE,
        EXPECTED_TEXTS - position
    )

    assert chunk.shape == (expected_rows, DIM)

    emb[position:position + len(chunk)] = chunk
    position += len(chunk)

    print(f"Copied chunk {i+1}/30 | Total rows: {position:,}")

emb.flush()
del emb

assert position == EXPECTED_TEXTS

# Atomically replace the final output
os.replace(TEMP_FILE, OUT_FILE)

# Final verification
final = np.load(OUT_FILE, mmap_mode="r")

assert final.shape == (EXPECTED_TEXTS, DIM)
assert final.dtype == np.float32

print("\n=== ASSEMBLY COMPLETE ===")
print("Shape:", final.shape)
print("Dtype:", final.dtype)
print("File size (GB):", round(OUT_FILE.stat().st_size / 1e9, 3))
print("Saved:", OUT_FILE)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")

# Load Dataset 3 unique texts and assembled embeddings
texts = pd.read_parquet(
    ROOT / "data_rep" / "external_unique_texts.parquet"
)
emb = np.load(
    ROOT / "data_rep" / "emb_external.npy",
    mmap_mode="r"
)

# Load original frozen centroids
centroids = np.load(
    ROOT / "data" / "narrative_centroids.npy"
)

print("=== DATASET 3 ===")
print("Unique text rows:", len(texts))
print("Columns:", texts.columns.tolist())
print("First rows:")
print(texts.head(3).to_string())

print("\n=== EMBEDDINGS ===")
print("Shape:", emb.shape)
print("Dtype:", emb.dtype)

print("\n=== ORIGINAL CENTROIDS ===")
print("Shape:", centroids.shape)
print("Dtype:", centroids.dtype)
print("All finite:", np.isfinite(centroids).all())
print("Centroid norms:", np.linalg.norm(centroids, axis=1))

# Inspect original narrative label convention
original_file = ROOT / "data" / "tweets_en.parquet"

if original_file.exists():
    original = pd.read_parquet(
        original_file,
        columns=["narrative"]
    )
    print("\n=== ORIGINAL NARRATIVE LABELS ===")
    print(original["narrative"].value_counts(dropna=False))
else:
    print("\nOriginal tweets_en.parquet not found at expected path.")

# Basic compatibility checks
assert len(texts) == emb.shape[0]
assert emb.shape[1] == centroids.shape[1]
assert centroids.shape[0] == 4
assert np.isfinite(centroids).all()

print("\n=== DIMENSION CHECK PASSED ===")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"

# Load inputs
texts = pd.read_parquet(
    REP / "external_unique_texts.parquet"
)

emb = np.load(
    REP / "emb_external.npy",
    mmap_mode="r"
)

centroids = np.load(
    ROOT / "data" / "narrative_centroids.npy"
)

# Verify compatibility
assert emb.shape == (len(texts), 384)
assert centroids.shape == (4, 384)
assert np.isfinite(centroids).all()

# Original assignment rule: cosine similarity via normalized vectors
# Both embeddings and centroids are already normalized.
similarities = np.asarray(emb @ centroids.T)

# Assign the nearest frozen centroid
narrative = similarities.argmax(axis=1).astype(np.int8)

# Confidence diagnostics
max_similarity = similarities.max(axis=1).astype(np.float32)

sorted_sim = np.sort(similarities, axis=1)
similarity_margin = (
    sorted_sim[:, -1] - sorted_sim[:, -2]
).astype(np.float32)

# Create labeled unique-text dataset
labeled = texts.copy()
labeled["narrative"] = narrative
labeled["narrative_similarity"] = max_similarity
labeled["narrative_margin"] = similarity_margin

OUT_FILE = REP / "external_unique_texts_labeled.parquet"
labeled.to_parquet(OUT_FILE, index=False)

# Report distribution
print("=== NARRATIVE ASSIGNMENT COMPLETE ===")
print("Unique texts:", len(labeled))
print("\nNarrative distribution:")
print(
    labeled["narrative"]
    .value_counts()
    .sort_index()
)

print("\nNarrative proportions:")
print(
    (labeled["narrative"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

print("\nSimilarity statistics:")
print(labeled["narrative_similarity"].describe())

print("\nSimilarity margin statistics:")
print(labeled["narrative_margin"].describe())

print("\nSaved:", OUT_FILE)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"

TWEETS_FILE = REP / "tweets_external_clean.parquet"
LABELS_FILE = REP / "external_unique_texts_labeled.parquet"
OUT_FILE = REP / "tweets_external_labeled.parquet"

# Load cleaned tweets and unique-text labels
tweets = pd.read_parquet(TWEETS_FILE)
labels = pd.read_parquet(
    LABELS_FILE,
    columns=[
        "text_idx",
        "narrative",
        "narrative_similarity",
        "narrative_margin"
    ]
)

print("Cleaned tweets:", len(tweets))
print("Unique text labels:", len(labels))

# Validate mapping keys
assert "text_idx" in tweets.columns
assert labels["text_idx"].is_unique
assert tweets["text_idx"].notna().all()
assert tweets["text_idx"].isin(labels["text_idx"]).all()

# Map each unique-text label to its corresponding tweets
result = tweets.merge(
    labels,
    on="text_idx",
    how="left",
    validate="many_to_one"
)

# Validate complete mapping
assert len(result) == len(tweets)
assert result["narrative"].notna().all()
assert result["narrative"].between(0, 3).all()

# Save separately; original cleaned data remains unchanged
result.to_parquet(OUT_FILE, index=False)

print("\n=== TWEET-LEVEL MAPPING COMPLETE ===")
print("Output rows:", len(result))
print("Unique tweet IDs:", result["tweet_id"].nunique())
print("\nNarrative distribution:")
print(result["narrative"].value_counts().sort_index())

print("\nNarrative proportions (%):")
print(
    (result["narrative"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

print("\nMissing narrative labels:", result["narrative"].isna().sum())
print("Saved:", OUT_FILE)


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"

FILE = REP / "tweets_external_labeled.parquet"

df = pd.read_parquet(FILE)

print("Columns:", df.columns.tolist())

# Parse timestamps
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")

assert df["date"].notna().all(), "Missing or invalid dates detected"

# Recalculate Monday-based weekly periods
df["week_check"] = (
    df["date"].dt.tz_localize(None)
      .dt.to_period("W-SUN")
      .dt.start_time
)

# Compare with existing week column, if available
if "week" in df.columns:
    existing_week = pd.to_datetime(df["week"], errors="coerce")
    matches = (
        existing_week.dt.normalize()
        == df["week_check"]
    )
    print("\nExisting week column agreement:", matches.mean())
    print("Week mismatches:", (~matches).sum())

# Weekly volume and unique channels
weekly = (
    df.groupby("week_check")
      .agg(
          tweet_count=("tweet_id", "count"),
          unique_tweets=("tweet_id", "nunique"),
          unique_channels=("channel", "nunique")
      )
)

# Narrative counts per week
narrative_counts = pd.crosstab(
    df["week_check"],
    df["narrative"]
).reindex(columns=[0, 1, 2, 3], fill_value=0)

narrative_counts.columns = [
    f"narrative_{i}_count" for i in range(4)
]

weekly = weekly.join(narrative_counts)

# Save weekly diagnostics
weekly = weekly.reset_index()
weekly["week"] = weekly["week_check"].dt.strftime("%Y-%m-%d")
weekly = weekly.drop(columns=["week_check"])

OUT = REP / "15_dataset3_weekly_coverage.csv"
weekly.to_csv(OUT, index=False)

print("\n=== WEEKLY COVERAGE ===")
print("Number of weeks:", len(weekly))
print("Total tweets:", weekly["tweet_count"].sum())
print("First week:", weekly["week"].min())
print("Last week:", weekly["week"].max())
print("Minimum weekly volume:", weekly["tweet_count"].min())
print("Maximum weekly volume:", weekly["tweet_count"].max())
print("Weeks below 30 tweets:", (weekly["tweet_count"] < 30).sum())

print("\nWeekly summary:")
print(weekly["tweet_count"].describe())

print("\nFirst 5 weeks:")
print(weekly.head().to_string(index=False))

print("\nLast 5 weeks:")
print(weekly.tail().to_string(index=False))

print("\nSaved:", OUT)


### 3.4 Build the Dataset 3 weekly target panel

The panel is built at narrative-week level. It includes semantic drift, hashtag-derived entity turnover, fragmentation, attention-share change, merging, volume growth, audience breadth, and engagement change. Geographic, stance, and sentiment targets remain unavailable where the source does not support them.

In [ ]:

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import entropy

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"

TWEETS_FILE = REP / "tweets_external_labeled.parquet"
EMB_FILE = REP / "emb_external.npy"

OUT_PANEL = REP / "panel_external.parquet"
OUT_CENTROIDS = REP / "centroids_external.npz"

MIN_TWEETS_CELL = 30

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

df = pd.read_parquet(TWEETS_FILE)
E = np.load(EMB_FILE, mmap_mode="r")

df["week"] = pd.to_datetime(df["week"]).dt.normalize()
df = df.reset_index(drop=True)

assert len(df) == 1_305_272
assert E.shape == (732_096, 384)
assert df["text_idx"].notna().all()

# Confirm text_idx is a valid embedding row index
assert df["text_idx"].between(0, E.shape[0] - 1).all()

# --------------------------------------------------
# 2. Define complete narrative-week grid
# --------------------------------------------------

weeks = pd.date_range(
    df["week"].min(),
    df["week"].max(),
    freq="7D"
)

nars = sorted(df["narrative"].dropna().unique())

K = len(nars)
W = len(weeks)
D = E.shape[1]

assert K == 4

# Group indices avoid repeatedly scanning the full dataset
grp = df.groupby(
    ["narrative", "week"],
    sort=False
).indices

cent = np.full((K, W, D), np.nan, dtype=np.float32)
raw = np.full_like(cent, np.nan)
nz = np.full((K, W), np.nan)

rows = []
tops = {}

# --------------------------------------------------
# 3. Narrative-week aggregation
# --------------------------------------------------

for ni, nar in enumerate(nars):

    for wi, week in enumerate(weeks):

        idx = grp.get((nar, week))

        if idx is None:
            idx = np.array([], dtype=np.int64)

        sub = df.iloc[idx]
        m = len(idx)

        rec = {
            "narrative": nar,
            "week": week,
            "n": m,
            "reach": np.nan,
            "sent": np.nan,
            "stance": np.nan,
            "pro_ua_share": np.nan,
            "pro_ru_share": np.nan,
            "country_entropy": np.nan,
            "verified_share": np.nan,
        }

        if m >= MIN_TWEETS_CELL:

            # Crucial: expand unique-text embeddings back to tweets
            # using text_idx, preserving tweet frequency.
            text_indices = sub["text_idx"].to_numpy(dtype=np.int64)
            cell_embeddings = E[text_indices]

            # Original weekly mean embedding
            c = cell_embeddings.mean(axis=0)
            raw[ni, wi] = c

            # Original sampling-noise correction
            nz[ni, wi] = (
                (1 - float(c @ c))
                * m / (m - 1) / m
            )

            # Normalized weekly narrative centroid
            cent[ni, wi] = c / (
                np.linalg.norm(c) + 1e-12
            )

            # Hashtag-derived entity frequencies
            ents = Counter()

            for ee in sub["entities"]:
                if not isinstance(ee, (list, tuple, np.ndarray)):
                    continue

                for e in ee:
                    if isinstance(e, (list, tuple, np.ndarray)):
                        if len(e) == 0:
                            continue
                        token = e[0]
                    else:
                        token = e

                    if token is not None and pd.notna(token):
                        ents[str(token).lower()] += 1

            tops[(nar, week)] = {
                key for key, _ in ents.most_common(20)
            }

            # User/channel distribution
            channel_counts = sub["channel"].dropna().value_counts()

            outlet_entropy = (
                float(entropy(channel_counts))
                if len(channel_counts) > 0
                else np.nan
            )

            rec.update({
                "ent_div": (
                    len(ents) / max(sum(ents.values()), 1)
                ),
                "n_channels": int(sub["channel"].nunique()),
                "outlet_entropy": outlet_entropy,

                # Original semantic dispersion formula
                "disp": (
                    (1 - float(c @ c))
                    * m / (m - 1)
                ),

                # Engagement is available as retweets + likes.
                "engage": float(
                    np.log1p(
                        sub["engagement"].fillna(0)
                    ).mean()
                ),
            })

        else:
            tops[(nar, week)] = None

            rec.update({
                "ent_div": np.nan,
                "n_channels": np.nan,
                "outlet_entropy": np.nan,
                "disp": np.nan,
                "engage": np.nan,
            })

        rows.append(rec)

    print(f"Processed narrative {nar} ({ni+1}/{K})")

# --------------------------------------------------
# 4. Create weekly panel
# --------------------------------------------------

P = (
    pd.DataFrame(rows)
    .sort_values(["narrative", "week"])
    .reset_index(drop=True)
)

# --------------------------------------------------
# 5. Semantic drift and bias-corrected drift
# --------------------------------------------------

dr = np.full((K, W), np.nan)
dd = np.full((K, W), np.nan)

dr[:, 1:] = 1 - np.einsum(
    "nwd,nwd->nw",
    cent[:, 1:],
    cent[:, :-1]
)

dd[:, 1:] = (
    ((raw[:, 1:] - raw[:, :-1]) ** 2).sum(axis=-1)
    - nz[:, 1:]
    - nz[:, :-1]
)

P["drift"] = dr.ravel()
P["drift_db"] = dd.ravel()

# --------------------------------------------------
# 6. Unsupported measures remain missing
# --------------------------------------------------

P["abs_sent_change"] = np.nan
P["abs_stance_change"] = np.nan

# --------------------------------------------------
# 7. Hashtag-derived entity turnover
# --------------------------------------------------

turnover = []

for nar, d in P.groupby("narrative", sort=True):

    previous = None

    for week in d["week"]:

        current = tops.get((nar, week))

        if current is None or previous is None:
            turnover.append(np.nan)
        else:
            turnover.append(
                1 - len(current & previous)
                / max(len(current | previous), 1)
            )

        previous = current

P["entity_turnover"] = turnover

# --------------------------------------------------
# 8. Volume, event week and attention share
# --------------------------------------------------

P["total_volume"] = (
    P.groupby("week")["n"]
    .transform("sum")
)

weekly_volume = (
    P.groupby("week")["total_volume"]
    .first()
)

trail = (
    weekly_volume.shift(1)
    .rolling(8, min_periods=4)
    .median()
)

event_week = (
    weekly_volume > 1.5 * trail
).astype(float)

P["event_week"] = P["week"].map(event_week)

P["share"] = (
    P["n"] / P["total_volume"].replace(0, np.nan)
)

# --------------------------------------------------
# 9. Structural measures
# --------------------------------------------------

max_sim = np.full((K, W), np.nan)

for wi in range(W):

    for ni in range(K):

        if not np.isfinite(cent[ni, wi, 0]):
            continue

        rival_sims = [
            float(cent[ni, wi] @ cent[mi, wi])
            for mi in range(K)
            if mi != ni
            and np.isfinite(cent[mi, wi, 0])
        ]

        if rival_sims:
            max_sim[ni, wi] = max(rival_sims)

P["max_sim"] = max_sim.ravel()

g = P.groupby("narrative")

P["abs_frag_change"] = (
    P["disp"] - g["disp"].shift(1)
).abs()

P["abs_share_change"] = (
    P["share"] - g["share"].shift(1)
).abs()

P["abs_merge_change"] = (
    P["max_sim"] - g["max_sim"].shift(1)
).abs()

# --------------------------------------------------
# 10. Diffusion measures
# --------------------------------------------------

P["d_volume"] = (
    np.log1p(P["n"])
    - np.log1p(g["n"].shift(1))
)

P["d_breadth"] = (
    P["outlet_entropy"]
    - g["outlet_entropy"].shift(1)
)

# Geographic information is unavailable
P["d_geo"] = np.nan

P["d_engage"] = (
    P["engage"] - g["engage"].shift(1)
)

# --------------------------------------------------
# 11. Save outputs
# --------------------------------------------------

P.to_parquet(OUT_PANEL, index=False)

np.savez(
    OUT_CENTROIDS,
    cent=cent,
    raw=raw,
    nz=nz,
    weeks=weeks.values.astype("datetime64[ns]"),
    narratives=np.array(nars)
)

# --------------------------------------------------
# 12. Validation and summary
# --------------------------------------------------

assert len(P) == K * W
assert P.groupby("narrative").size().eq(W).all()
assert P["n"].sum() == len(df)

print("\n=== DATASET 3 TARGET PANEL COMPLETE ===")

print("Narratives:", K)
print("Weeks:", W)
print("Panel rows:", len(P))
print("Total tweets:", int(P["n"].sum()))

print("\nAvailable target counts:")
for col in [
    "drift_db",
    "entity_turnover",
    "abs_frag_change",
    "abs_share_change",
    "abs_merge_change",
    "d_volume",
    "d_breadth",
    "d_geo",
    "d_engage",
]:
    print(f"{col:24s}: {P[col].notna().sum():,}")

print("\nTarget medians:")
print(
    P.groupby("narrative")[
        [
            "n",
            "drift_db",
            "entity_turnover",
            "abs_frag_change",
            "abs_share_change",
            "abs_merge_change",
            "d_volume",
            "d_breadth",
        ]
    ].median().round(4)
)

print("\nSaved panel:", OUT_PANEL)
print("Saved centroids:", OUT_CENTROIDS)


### 3.5 Construct reduced competition pressure

The three available components are:
1. Semantic similarity between focal and rival narrative centroids.
2. Relative narrative growth.
3. Shared-user network proximity based on user-count vectors.

The equal-weight reduced score is named `pressure_eq_reduced`; it is not the original six-component score or CPA measure.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from sklearn.preprocessing import normalize

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
REP = ROOT / "data_rep"

PANEL_FILE = REP / "panel_external.parquet"
CENTROID_FILE = REP / "centroids_external.npz"
TWEETS_FILE = REP / "tweets_external_labeled.parquet"

OUT_FILE = REP / "panel_pressure_external.parquet"
DIAG_FILE = REP / "23_external_pressure_diagnostics.csv"

# --------------------------------------------------
# 1. Load weekly panel and centroids
# --------------------------------------------------

P = pd.read_parquet(PANEL_FILE)
P["week"] = pd.to_datetime(P["week"])

Z = np.load(CENTROID_FILE)

cent = Z["cent"]
weeks = pd.to_datetime(Z["weeks"])
nars = list(Z["narratives"])

K, W = cent.shape[:2]

assert K == 4
assert len(P) == K * W

P = P.sort_values(
    ["week", "narrative"]
).reset_index(drop=True)

def mat(col):
    return (
        P.pivot(
            index="narrative",
            columns="week",
            values=col
        )
        .reindex(index=nars, columns=weeks)
        .to_numpy(float)
    )

n = mat("n")

# Weekly relative growth, same definition as original
growth = np.full((K, W), np.nan)

growth[:, 1:] = (
    np.log1p(n[:, 1:])
    - np.log1p(n[:, :-1])
)

# --------------------------------------------------
# 2. Build sparse user/channel distribution matrix
# --------------------------------------------------

# Load only the required columns to reduce RAM usage
T = pd.read_parquet(
    TWEETS_FILE,
    columns=["narrative", "week", "channel"]
)

T["week"] = pd.to_datetime(T["week"]).dt.normalize()

T = T.dropna(
    subset=["narrative", "week", "channel"]
)

# Map each observation to its narrative-week row
week_map = {
    week: i for i, week in enumerate(weeks)
}

nar_map = {
    nar: i for i, nar in enumerate(nars)
}

T["wi"] = T["week"].map(week_map)
T["ni"] = T["narrative"].map(nar_map)

T = T.dropna(subset=["wi", "ni"])

# Count tweets per user/channel in each narrative-week
counts = (
    T.groupby(["wi", "ni", "channel"], observed=True)
     .size()
     .reset_index(name="count")
)

# Convert user/channel identifiers to integer columns
channel_codes, unique_channels = pd.factorize(
    counts["channel"],
    sort=False
)

row_codes = (
    counts["wi"].to_numpy(dtype=np.int64) * K
    + counts["ni"].to_numpy(dtype=np.int64)
)

X = coo_matrix(
    (
        counts["count"].to_numpy(dtype=np.float64),
        (
            row_codes,
            channel_codes
        )
    ),
    shape=(W * K, len(unique_channels))
).tocsr()

# L2-normalize each narrative-week user distribution
X = normalize(X, norm="l2", axis=1)

print("Sparse user matrix:", X.shape)
print("Non-zero user counts:", X.nnz)

# --------------------------------------------------
# 3. Calculate pairwise network cosine similarities
# --------------------------------------------------

network = np.full(
    (W, K, K),
    np.nan,
    dtype=np.float32
)

for wi in range(W):

    for ni in range(K):

        row_a = wi * K + ni

        for mi in range(K):

            if ni == mi:
                continue

            row_b = wi * K + mi

            similarity = X.getrow(row_a).multiply(
                X.getrow(row_b)
            ).sum()

            network[wi, ni, mi] = float(similarity)

# --------------------------------------------------
# 4. Calculate three competition components
# --------------------------------------------------

records = []

for wi, week in enumerate(weeks):

    for ni, nar in enumerate(nars):

        if not np.isfinite(cent[ni, wi, 0]):
            continue

        rivals = [
            mi for mi in range(K)
            if mi != ni
            and np.isfinite(cent[mi, wi, 0])
        ]

        if not rivals:
            continue

        weights = n[rivals, wi]

        # Semantic relation
        semantic = np.array([
            max(
                0.0,
                float(cent[ni, wi] @ cent[mi, wi])
            )
            for mi in rivals
        ])

        # Relative growth of competing narratives
        relative_growth = np.array([
            growth[mi, wi] - growth[ni, wi]
            for mi in rivals
        ])

        # User/channel overlap proxy
        net = np.array([
            network[wi, ni, mi]
            for mi in rivals
        ])

        def weighted_average(values, weights):
            valid = (
                np.isfinite(values)
                & np.isfinite(weights)
                & (weights > 0)
            )

            if not valid.any():
                return np.nan

            return float(
                np.average(
                    values[valid],
                    weights=weights[valid]
                )
            )

        records.append({
            "narrative": nar,
            "week": week,
            "comp_sem": weighted_average(
                semantic, weights
            ),
            "comp_growth": weighted_average(
                relative_growth, weights
            ),
            "comp_network": weighted_average(
                net, weights
            ),
        })

C = pd.DataFrame(records)

# --------------------------------------------------
# 5. Expanding standardization
# --------------------------------------------------

# Preserve raw values and create separately named z-scores.
# Same expanding convention as the original script:
# statistics use observations with week <= current week.

components = [
    "comp_sem",
    "comp_growth",
    "comp_network"
]

for col in components:

    z_col = f"{col}_z"
    C[z_col] = np.nan

    for week in sorted(C["week"].unique()):

        past = C.loc[
            C["week"] <= week,
            col
        ].dropna()

        current = C["week"] == week

        if len(past) < 5 or past.std() == 0:
            C.loc[current, z_col] = 0.0
        else:
            C.loc[current, z_col] = (
                C.loc[current, col] - past.mean()
            ) / past.std()

# Reduced score: equal-weight mean of the three available z-scores
z_cols = [f"{c}_z" for c in components]

C["pressure_eq_reduced"] = C[z_cols].mean(
    axis=1,
    skipna=True
)

# --------------------------------------------------
# 6. Explicitly unavailable original components
# --------------------------------------------------

C["comp_stance"] = np.nan
C["comp_reach"] = np.nan
C["comp_overlap"] = np.nan
C["cpa"] = np.nan

# --------------------------------------------------
# 7. Merge with external target panel
# --------------------------------------------------

P = P.merge(
    C,
    on=["narrative", "week"],
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------
# 8. Validation
# --------------------------------------------------

assert len(P) == K * W
assert P["comp_sem"].notna().any()
assert P["comp_network"].notna().any()
assert P["pressure_eq_reduced"].notna().any()

assert P["comp_stance"].isna().all()
assert P["comp_reach"].isna().all()
assert P["comp_overlap"].isna().all()
assert P["cpa"].isna().all()

# Save separately
P.to_parquet(OUT_FILE, index=False)

# Component diagnostics
diag = (
    P.groupby("narrative")[
        [
            "comp_sem",
            "comp_growth",
            "comp_network",
            "pressure_eq_reduced"
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
)

diag.to_csv(DIAG_FILE)

print("\n=== REDUCED PRESSURE COMPLETE ===")
print("Panel rows:", len(P))

print("\nAvailable component counts:")
for col in components + ["pressure_eq_reduced"]:
    print(
        f"{col:24s}: "
        f"{P[col].notna().sum():,}"
    )

print("\nUnavailable components:")
for col in [
    "comp_stance",
    "comp_reach",
    "comp_overlap",
    "cpa"
]:
    print(f"{col:24s}: {P[col].notna().sum():,}")

print("\nReduced pressure summary:")
print(
    P.groupby("narrative")[
        components + ["pressure_eq_reduced"]
    ].mean().round(4)
)

print("\nSaved panel:", OUT_FILE)
print("Saved diagnostics:", DIAG_FILE)


## 4. Dataset 3 predictive evaluation

### 4.1 Baseline models
The adapted external baseline uses only available predictors. Sentiment, stance, follower reach, country, and verification are excluded rather than imputed as fabricated values.

In [ ]:

%%writefile "/content/drive/MyDrive/ADA Jackfruit/24_external_baseline.py"

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
DATA = ROOT / "data_rep"
RESULTS = ROOT / "results_rep"

RESULTS.mkdir(parents=True, exist_ok=True)

PANEL = DATA / "panel_pressure_external.parquet"

TEST_START = pd.Timestamp("2023-01-01")
DELTA = 1
MIN_TRAIN = 40
SEED = 42

TARGETS = [
    "drift_db",
    "entity_turnover",
    "abs_frag_change",
    "abs_share_change",
    "abs_merge_change",
    "d_volume",
    "d_breadth",
    "d_engage",
]

MODELS = {
    "mean": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        DummyRegressor(strategy="mean")
    ),

    "ridge": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "hgb": lambda: HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_depth=3,
        l2_regularization=1.0,
        random_state=SEED
    ),
}


def add_external_features(panel, target):
    df = (
        panel.sort_values(["narrative", "week"])
        .reset_index(drop=True)
        .copy()
    )

    g = df.groupby("narrative")[target]

    df["h_lag1"] = g.shift(1)
    df["h_lag2"] = g.shift(2)

    df["h_roll4"] = g.transform(
        lambda s: s.rolling(4, min_periods=2).mean()
    )

    df["log_n"] = np.log1p(df["n"])
    df["log_channels"] = np.log1p(df["n_channels"])
    df["log_total_volume"] = np.log1p(df["total_volume"])

    df["y"] = g.shift(-DELTA)

    dummies = pd.get_dummies(
        df["narrative"],
        prefix="nar",
        dtype=float
    )

    return pd.concat([df, dummies], axis=1)


def external_baseline_features(df):
    return [
        "narrative_target",
        "h_lag1",
        "h_lag2",
        "h_roll4",
        "ent_div",
        "log_n",
        "log_channels",
        "log_total_volume",
        "event_week",
    ] + [
        c for c in df.columns
        if c.startswith("nar_")
    ]


def walk_forward(df, features, make_model):
    test_weeks = [
        w for w in sorted(df.week.unique())
        if w >= TEST_START
    ]

    predictions = pd.Series(np.nan, index=df.index)
    model = None

    for i, t in enumerate(test_weeks):

        test = df[
            (df.week == t) &
            df["y"].notna()
        ]

        if test.empty:
            continue

        train = df[
            (df.week <= t - pd.Timedelta(weeks=DELTA)) &
            df["y"].notna()
        ]

        if len(train) < MIN_TRAIN:
            continue

        model = make_model()

        model.fit(
            train[features],
            train["y"]
        )

        predictions.loc[test.index] = model.predict(
            test[features]
        )

    return predictions


def persistence(df, target):
    return df[target].where(df.week >= TEST_START)


def score(df, predictions):
    idx = None

    for pred in predictions.values():
        valid = pred.dropna().index
        idx = valid if idx is None else idx.intersection(valid)

    y = df.loc[idx, "y"]

    rows = []

    for name, pred in predictions.items():
        error = y - pred.loc[idx]

        rows.append({
            "model": name,
            "n": len(idx),
            "mae": error.abs().mean(),
            "mse": (error ** 2).mean()
        })

    result = pd.DataFrame(rows).set_index("model")

    if "persistence" in result.index:
        result["skill_vs_persist"] = (
            1 - result.mse / result.loc["persistence", "mse"]
        )

    return result, idx


def main():

    print("Loading Dataset 3 pressure panel...")
    panel = pd.read_parquet(PANEL)

    print("Panel rows:", len(panel))

    required = [
        "narrative", "week", "n", "n_channels",
        "total_volume", "ent_div", "event_week"
    ]

    missing = [c for c in required if c not in panel.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    all_results = []

    for target in TARGETS:

        print("\n" + "=" * 75)
        print("TARGET:", target)
        print("=" * 75)

        df = add_external_features(panel, target)

        # Current target is retained as an explicit historical predictor,
        # matching the original baseline's use of the current target.
        df["narrative_target"] = df[target]

        features = external_baseline_features(df)

        predictions = {
            "persistence": persistence(df, target)
        }

        predictions["train_mean"] = walk_forward(
            df, ["h_lag1"], MODELS["mean"]
        )

        for model_name in ["ridge", "hgb"]:
            predictions[model_name] = walk_forward(
                df, features, MODELS[model_name]
            )

        result, idx = score(df, predictions)

        print(
            f"Held-out narrative-weeks: {len(idx)}"
        )
        print(result.round(5))

        result = result.assign(target=target)
        all_results.append(result)

    combined = pd.concat(all_results)

    output = RESULTS / "24_external_baseline.csv"
    combined.to_csv(output)

    print("\n=== EXTERNAL BASELINE COMPLETE ===")
    print("Saved:", output)


if __name__ == "__main__":
    main()


In [ ]:
%run "/content/drive/MyDrive/ADA Jackfruit/24_external_baseline.py"

### 4.2 Corrected competition-feature evaluation

**Authoritative external test:** Step 27. Both the baseline and pressure-augmented models use the same four-week refitting schedule; placebo models use that same schedule. The test covers eight outcomes and two models, with 500 block-shuffled placebos and 1,000 paired bootstrap resamples.

Do not use the earlier Step 25 and Step 26 results for inference: their real-model and placebo refit schedules did not match.


In [ ]:

%%writefile "/content/drive/MyDrive/ADA Jackfruit/27_external_corrected_placebo.py"

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
DATA = ROOT / "data_rep"
RESULTS = ROOT / "results_rep"

RESULTS.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "drift_db",
    "entity_turnover",
    "abs_frag_change",
    "abs_share_change",
    "abs_merge_change",
    "d_volume",
    "d_breadth",
    "d_engage"
]

PRESSURE = [
    "comp_sem_z",
    "comp_growth_z",
    "comp_network_z"
]

TEST_START = pd.Timestamp("2023-01-01")
DELTA = 1
MIN_TRAIN = 40
REFIT_EVERY = 4

N_PLACEBO = 500
N_BOOT = 1000
BLOCK = 4
SEED = 42

MODELS = {
    "ridge": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "hgb": lambda: HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_depth=3,
        l2_regularization=1.0,
        random_state=SEED
    )
}


def prepare_data(panel, target):

    df = (
        panel.sort_values(["narrative", "week"])
        .reset_index(drop=True)
        .copy()
    )

    g = df.groupby("narrative")[target]

    df["h_lag1"] = g.shift(1)
    df["h_lag2"] = g.shift(2)

    df["h_roll4"] = g.transform(
        lambda s: s.rolling(4, min_periods=2).mean()
    )

    df["log_n"] = np.log1p(df["n"])
    df["log_channels"] = np.log1p(df["n_channels"])
    df["log_total_volume"] = np.log1p(df["total_volume"])

    df["y"] = g.shift(-DELTA)

    dummies = pd.get_dummies(
        df["narrative"],
        prefix="nar",
        dtype=float
    )

    df = pd.concat([df, dummies], axis=1)

    baseline = [
        target,
        "h_lag1",
        "h_lag2",
        "h_roll4",
        "ent_div",
        "log_n",
        "log_channels",
        "log_total_volume",
        "event_week"
    ] + [
        c for c in df.columns if c.startswith("nar_")
    ]

    return df, baseline


def walk_forward(
    df,
    features,
    model_factory,
    refit_every=REFIT_EVERY
):

    test_weeks = [
        w for w in sorted(df.week.unique())
        if w >= TEST_START
    ]

    predictions = pd.Series(np.nan, index=df.index)
    model = None

    for i, t in enumerate(test_weeks):

        test = df[
            (df.week == t) &
            df["y"].notna()
        ]

        if test.empty:
            continue

        if model is None or i % refit_every == 0:

            train = df[
                (df.week <= t - pd.Timedelta(weeks=DELTA)) &
                df["y"].notna()
            ]

            if len(train) < MIN_TRAIN:
                continue

            model = model_factory()

            model.fit(
                train[features],
                train["y"]
            )

        if model is not None:
            predictions.loc[test.index] = model.predict(
                test[features]
            )

    return predictions


def block_perm(length, block, rng):

    blocks = [
        np.arange(i, min(i + block, length))
        for i in range(0, length, block)
    ]

    rng.shuffle(blocks)

    return np.concatenate(blocks)


def shuffle_cols(df, cols, block, rng):

    out = df.copy()
    values = df[cols].to_numpy().copy()
    narratives = df["narrative"].to_numpy()

    for narrative in np.unique(narratives):

        positions = np.where(narratives == narrative)[0]

        values[positions] = values[positions][
            block_perm(len(positions), block, rng)
        ]

    out[cols] = values

    return out


def mse(df, prediction, idx):

    return float(
        (
            df.loc[idx, "y"] -
            prediction.loc[idx]
        ).pow(2).mean()
    )


def main():

    rng = np.random.default_rng(SEED)

    panel = pd.read_parquet(
        DATA / "panel_pressure_external.parquet"
    )

    all_records = []

    output = RESULTS / "27_external_corrected_placebo.csv"

    for target in TARGETS:

        print("\n" + "=" * 85)
        print("TARGET:", target)
        print("=" * 85)

        df, baseline = prepare_data(panel, target)

        for model_name, model_factory in MODELS.items():

            print("\nMODEL:", model_name)

            # Both real models use four-week refitting.
            base_pred = walk_forward(
                df,
                baseline,
                model_factory,
                refit_every=REFIT_EVERY
            )

            pressure_pred = walk_forward(
                df,
                baseline + PRESSURE,
                model_factory,
                refit_every=REFIT_EVERY
            )

            idx = base_pred.dropna().index.intersection(
                pressure_pred.dropna().index
            )

            if len(idx) == 0:
                raise RuntimeError(
                    f"No common held-out rows for {target}, {model_name}"
                )

            base_mse = mse(df, base_pred, idx)
            pressure_mse = mse(df, pressure_pred, idx)

            real_gain = 1 - pressure_mse / base_mse

            # Paired bootstrap over held-out weeks.
            weekly = pd.DataFrame({
                "week": df.loc[idx, "week"],
                "difference": (
                    (df.loc[idx, "y"] - base_pred.loc[idx]) ** 2
                    -
                    (df.loc[idx, "y"] - pressure_pred.loc[idx]) ** 2
                )
            }).groupby("week")["difference"].mean().to_numpy()

            bootstrap = np.array([
                rng.choice(
                    weekly,
                    size=len(weekly),
                    replace=True
                ).mean()
                for _ in range(N_BOOT)
            ])

            ci_lo, ci_hi = np.percentile(
                bootstrap, [2.5, 97.5]
            )

            print("Held-out rows:", len(idx))
            print("Four-week baseline MSE:", base_mse)
            print("Four-week pressure MSE:", pressure_mse)
            print("Real gain:", round(real_gain, 6))
            print(
                "Paired bootstrap 95% CI:",
                (round(ci_lo, 8), round(ci_hi, 8))
            )

            # Placebo: shuffle all three pressure components.
            placebo_gains = []

            for iteration in range(N_PLACEBO):

                shuffled = shuffle_cols(
                    df,
                    PRESSURE,
                    BLOCK,
                    rng
                )

                placebo_pred = walk_forward(
                    shuffled,
                    baseline + PRESSURE,
                    model_factory,
                    refit_every=REFIT_EVERY
                )

                if placebo_pred.loc[idx].isna().any():
                    raise RuntimeError(
                        "Placebo prediction coverage differs from real coverage"
                    )

                placebo_mse = mse(
                    shuffled,
                    placebo_pred,
                    idx
                )

                placebo_gain = 1 - placebo_mse / base_mse

                placebo_gains.append(placebo_gain)

                if (iteration + 1) % 100 == 0:
                    print(
                        f"Placebo progress: "
                        f"{iteration + 1}/{N_PLACEBO}"
                    )

            placebo_gains = np.asarray(placebo_gains)

            placebo_p = (
                1 + np.sum(placebo_gains >= real_gain)
            ) / (1 + N_PLACEBO)

            placebo_mean = placebo_gains.mean()
            placebo_p95 = np.percentile(placebo_gains, 95)

            print("\nCORRECTED PLACEBO RESULT")
            print("Real gain:", round(real_gain, 6))
            print("Placebo mean:", round(placebo_mean, 6))
            print("Placebo 95th percentile:", round(placebo_p95, 6))
            print("One-sided placebo p:", round(placebo_p, 6))

            all_records.append({
                "target": target,
                "model": model_name,
                "n_test": len(idx),
                "refit_every": REFIT_EVERY,
                "baseline_mse": base_mse,
                "pressure_mse": pressure_mse,
                "real_gain": real_gain,
                "bootstrap_ci_lo": ci_lo,
                "bootstrap_ci_hi": ci_hi,
                "placebo_mean": placebo_mean,
                "placebo_p95": placebo_p95,
                "placebo_p": placebo_p
            })

        # Save a checkpoint after every completed outcome.
        pd.DataFrame(all_records).to_csv(
            output,
            index=False
        )

        print("\nCheckpoint saved:", output)

    print("\n=== CORRECTED EXTERNAL PLACEBO TESTS COMPLETE ===")
    print("Final results:", output)


if __name__ == "__main__":
    main()


In [ ]:
%run "/content/drive/MyDrive/ADA Jackfruit/27_external_corrected_placebo.py"

### 4.3 Holm multiple-testing correction

Holm correction is applied to the 16 model–outcome placebo p-values from the corrected external test. This is a separate test family from the primary study's correction family.

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")
INPUT = ROOT / "results_rep/27_external_corrected_placebo.csv"
OUTPUT = ROOT / "results_rep/28_external_holm_correction.csv"

df = pd.read_csv(INPUT)

# Validate the expected results
assert len(df) == 16, f"Expected 16 tests, found {len(df)}"
assert "placebo_p" in df.columns, f"Missing placebo_p. Columns: {list(df.columns)}"

# Holm-Bonferroni correction
p = df["placebo_p"].to_numpy(dtype=float)
assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()

order = np.argsort(p)
m = len(p)
adjusted_sorted = np.maximum.accumulate(
    (m - np.arange(m)) * p[order]
)
adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

holm = np.empty(m)
holm[order] = adjusted_sorted

df["holm_p"] = holm
df["significant_holm_05"] = df["holm_p"] < 0.05

df = df.sort_values("placebo_p").reset_index(drop=True)
df.to_csv(OUTPUT, index=False)

print("=== HOLM CORRECTION COMPLETE ===")
print(df[[
    "target", "model", "real_gain",
    "placebo_p", "holm_p", "significant_holm_05"
]].to_string(index=False))

print("\nSignificant tests after Holm correction:",
      int(df["significant_holm_05"].sum()), "/", len(df))
print("Saved:", OUTPUT)


## 5. Final output validation and figures

The following checks inspect the final result files and confirm the expected test counts. They do not rerun model training.

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ADA Jackfruit")

files = {
    "Baseline": ROOT / "results_rep/24_external_baseline.csv",
    "Corrected placebo": ROOT / "results_rep/27_external_corrected_placebo.csv",
    "Holm correction": ROOT / "results_rep/28_external_holm_correction.csv",
}

for name, path in files.items():
    print(f"\n{'='*55}")
    print(name)

    if not path.exists():
        print("MISSING:", path)
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("Missing values:", int(df.isna().sum().sum()))

    if name == "Corrected placebo":
        print("Expected rows: 16")
        print("Unique outcomes:", df["target"].nunique())
        print("Unique models:", df["model"].nunique())

    if name == "Holm correction":
        print("Expected rows: 16")
        print("Significant after Holm:",
              int(df["significant_holm_05"].sum()))

print("\nVerification finished. No files modified.")


In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ADA Jackfruit/"
    "results_rep/24_external_baseline.csv"
)

df = pd.read_csv(path)

print("Models:")
print(df["model"].value_counts().to_string())

print("\nTargets:")
print(df["target"].value_counts().to_string())

print("\nComplete baseline table:")
print(df.sort_values(["target", "model"]).to_string(index=False))


### 5.1 Dataset 3 figures

The six final figures are generated by the separate script `generate_dataset3_figures.py`. Keep that script in the project folder and run it after the result CSVs and Parquet files are present.

Expected outputs under `figs_rep/`:
- `figure_1_acquisition_flow.png`
- `figure_2_weekly_tweet_volume.png`
- `figure_3_narrative_distribution.png`
- `figure_4_baseline_comparison.png`
- `figure_5_competition_gains.png`
- `figure_6_placebo_holm_pvalues.png`


In [ ]:
from pathlib import Path
import subprocess
import sys

FIGURE_SCRIPT = ROOT / "generate_dataset3_figures.py"
if FIGURE_SCRIPT.exists():
    subprocess.run([sys.executable, str(FIGURE_SCRIPT)], check=True)
else:
    print("Figure script not found:", FIGURE_SCRIPT)
    print("Upload/copy generate_dataset3_figures.py into the project root, then rerun this cell.")
